In [18]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

In [26]:
orders_df = pd.read_csv("orders.csv")
orders_df

,Order_ID,Order_Date,Customer_Segment,Priority,Product_Category,Order_Value_INR,Origin,Destination,Special_Handling
0,ORD000001,2025-10-09,Individual,Express,Industrial,238.73,Kolkata,Hyderabad,NaN
1,ORD000002,2025-09-29,SMB,Express,Industrial,17.01,Hyderabad,Kolkata,NaN
2,ORD000003,2025-09-15,SMB,Economy,Industrial,3024.95,Mumbai,Pune,NaN
3,ORD000004,2025-10-13,Individual,Economy,Fashion,56.74,Hyderabad,Ahmedabad,NaN
4,ORD000005,2025-09-08,SMB,Standard,Fashion,19148.65,Chennai,Mumbai,NaN
...,...,...,...,...,...,...,...,...,...
195,ORD000196,2025-10-07,Enterprise,Economy,Fashion,442.46,Mumbai,Chennai,NaN
196,ORD000197,2025-10-03,SMB,Economy,Books,31.82,Delhi,Kolkata,NaN
197,ORD000198,2025-10-12,Individual,Economy,Healthcare,104.50,Mumbai,Dubai,NaN
198,ORD000199,2025-10-14,Enterprise,Economy,Electronics,570.44,Pune,Ahmedabad,Fragile


In [27]:
orders_df.dtypes

Order_ID             object
Order_Date           object
Customer_Segment     object
Priority             object
Product_Category     object
Order_Value_INR     float64
Origin               object
Destination          object
Special_Handling     object
dtype: object

## encoding columns

In [67]:
def preprocess_orders(orders_df):
    # convert order_date to datetime
    orders_df['Order_Date'] = pd.to_datetime(orders_df['Order_Date'])
    
    # Convert categorical columns with proper encoding
    categorical_cols = ['Customer_Segment', 'Priority', 'Product_Category', 'Origin', 'Destination', 'Special_Handling']
    for col in categorical_cols:
        orders_df[col] = orders_df[col].astype('category')
        print(f"{col}: {orders_df[col].nunique()} unique values")
    
    return orders_df  

orders_df = preprocess_orders(orders_df)

Customer_Segment: 3 unique values
Priority: 3 unique values
Product_Category: 7 unique values
Origin: 8 unique values
Destination: 12 unique values
Special_Handling: 8 unique values


In [33]:
orders_df.dtypes

Order_ID                    object
Order_Date          datetime64[ns]
Customer_Segment          category
Priority                  category
Product_Category          category
Order_Value_INR            float64
Origin                    category
Destination               category
Special_Handling          category
dtype: object

## checking unique values

In [36]:
orders_df['Special_Handling'].unique()

[NaN, 'Hazmat', 'Fragile', 'Temperature_Controlled']
Categories (3, object): ['Fragile', 'Hazmat', 'Temperature_Controlled']

In [50]:
recent_date = orders_df['Order_Date'].max()
past_date = orders_df['Order_Date'].min()
print(recent_date)
print(past_date)

2025-10-20 00:00:00
2025-09-01 00:00:00


In [38]:
orders_df['Customer_Segment'].unique()

['Individual', 'SMB', 'Enterprise']
Categories (3, object): ['Enterprise', 'Individual', 'SMB']

In [41]:
orders_df['Priority'].unique()

['Express', 'Economy', 'Standard']
Categories (3, object): ['Economy', 'Express', 'Standard']

In [42]:
orders_df['Product_Category'].unique()

['Industrial', 'Fashion', 'Food & Beverage', 'Electronics', 'Books', 'Healthcare', 'Home Goods']
Categories (7, object): ['Books', 'Electronics', 'Fashion', 'Food & Beverage', 'Healthcare', 'Home Goods', 'Industrial']

In [48]:
max_order = orders_df['Order_Value_INR'].max()
min_order = orders_df['Order_Value_INR'].min()
print(max_order)
print(min_order)

47177.07
1.91


In [45]:
orders_df['Origin'].unique()

['Kolkata', 'Hyderabad', 'Mumbai', 'Chennai', 'Pune', 'Delhi', 'Ahmedabad', 'Bangalore']
Categories (8, object): ['Ahmedabad', 'Bangalore', 'Chennai', 'Delhi', 'Hyderabad', 'Kolkata', 'Mumbai', 'Pune']

In [46]:
orders_df['Destination'].unique()

['Hyderabad', 'Kolkata', 'Pune', 'Ahmedabad', 'Mumbai', ..., 'Hong Kong', 'Bangkok', 'Delhi', 'Singapore', 'Chennai']
Length: 12
Categories (12, object): ['Ahmedabad', 'Bangalore', 'Bangkok', 'Chennai', ..., 'Kolkata', 'Mumbai', 'Pune', 'Singapore']

## checking null values

In [56]:
orders_df.isnull().sum()

Order_ID              0
Order_Date            0
Customer_Segment      0
Priority              0
Product_Category      0
Order_Value_INR       0
Origin                0
Destination           0
Special_Handling    153
dtype: int64

In [24]:
print("Dataset Info:")
print(data.info())
print("\nDataset Shape:", data.shape)
print("\nFirst 5 rows:")
print(data.head())
print("\nMissing Values:")
print(data.isnull().sum())
print("\nBasic Statistics:")
print(data.describe())

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Order_ID          200 non-null    object 
 1   Order_Date        200 non-null    object 
 2   Customer_Segment  200 non-null    object 
 3   Priority          200 non-null    object 
 4   Product_Category  200 non-null    object 
 5   Order_Value_INR   200 non-null    float64
 6   Origin            200 non-null    object 
 7   Destination       200 non-null    object 
 8   Special_Handling  200 non-null    object 
dtypes: float64(1), object(8)
memory usage: 14.2+ KB
None

Dataset Shape: (200, 9)

First 5 rows:
    Order_ID  Order_Date Customer_Segment  Priority Product_Category  \
0  ORD000001  2025-10-09       Individual   Express       Industrial   
1  ORD000002  2025-09-29              SMB   Express       Industrial   
2  ORD000003  2025-09-15              SMB   Economy

## Cleaning Special_Handling Column

In [66]:
def fix_special_handling(orders_df):
    df = orders_df.copy()
    df['Special_Handling'] = df['Special_Handling'].astype(str)
    df['Special_Handling'] = df['Special_Handling'].replace('nan', np.nan)
    
    # Rule 1: High value orders

    high_value_threshold = df['Order_Value_INR'].quantile(0.75)
    high_value_mask = (df['Special_Handling'].isna()) & (df['Order_Value_INR'] > high_value_threshold)
    df.loc[high_value_mask, 'Special_Handling'] = 'High_Value_Care'
    
    # Rule 2: Express priority
    express_mask = (df['Special_Handling'].isna()) & (df['Priority'] == 'Express')
    df.loc[express_mask, 'Special_Handling'] = 'Express_Care'
    
    # Rule 3: International shipments
    international_mask = (df['Special_Handling'].isna()) & (df['Destination'] == 'Dubai')
    df.loc[international_mask, 'Special_Handling'] = 'International'
    
    # Rule 4: Product-based
    product_rules = {'Electronics': 'Fragile', 'Healthcare': 'Temperature_Controlled', 'Industrial': 'Heavy_Equipment'}
    for product, handling in product_rules.items():
        product_mask = (df['Special_Handling'].isna()) & (df['Product_Category'] == product)
        df.loc[product_mask, 'Special_Handling'] = handling
    
    # fill remaining NaN with 'Standard'
    df['Special_Handling'] = df['Special_Handling'].fillna('Standard')
    
    df['Special_Handling'] = df['Special_Handling'].astype('category')
    return df

orders_df = fix_special_handling(orders_df)

In [62]:
orders_df.head()

,Order_ID,Order_Date,Customer_Segment,Priority,Product_Category,Order_Value_INR,Origin,Destination,Special_Handling
0,ORD000001,2025-10-09,Individual,Express,Industrial,238.73,Kolkata,Hyderabad,Express_Care
1,ORD000002,2025-09-29,SMB,Express,Industrial,17.01,Hyderabad,Kolkata,Express_Care
2,ORD000003,2025-09-15,SMB,Economy,Industrial,3024.95,Mumbai,Pune,High_Value_Care
3,ORD000004,2025-10-13,Individual,Economy,Fashion,56.74,Hyderabad,Ahmedabad,Standard
4,ORD000005,2025-09-08,SMB,Standard,Fashion,19148.65,Chennai,Mumbai,High_Value_Care


## here we go..!!

In [63]:
orders_df.isnull().sum()

Order_ID            0
Order_Date          0
Customer_Segment    0
Priority            0
Product_Category    0
Order_Value_INR     0
Origin              0
Destination         0
Special_Handling    0
dtype: int64

## data is cleaned successfully ✅ 

In [69]:
# save as cleaned csv
orders_df.to_csv('cleaned_orders.csv', index=False)

# 2

### ROUTES_DISTANCE

In [71]:
routes_df = pd.read_csv("routes_distance.csv")
routes_df

,Order_ID,Route,Distance_KM,Fuel_Consumption_L,Toll_Charges_INR,Traffic_Delay_Minutes,Weather_Impact
0,ORD000001,Kolkata-Hyderabad,152.59,23.02,122.08,21,NaN
1,ORD000002,Hyderabad-Kolkata,362.05,43.98,289.64,33,NaN
2,ORD000003,Mumbai-Pune,519.74,65.75,415.79,2,NaN
3,ORD000004,Hyderabad-Ahmedabad,540.87,61.85,432.70,112,NaN
4,ORD000005,Chennai-Mumbai,1251.56,147.54,1001.25,10,NaN
...,...,...,...,...,...,...,...
145,ORD000146,Delhi-Bangalore,1361.22,166.83,1088.98,86,NaN
146,ORD000147,Hyderabad-Chennai,1623.27,197.97,1298.62,111,Light_Rain
147,ORD000148,Delhi-Mumbai,408.74,47.76,326.99,46,Light_Rain
148,ORD000149,Bangalore-Chennai,1774.26,215.45,1419.41,84,NaN


In [72]:
routes_df.isnull().sum()

Order_ID                   0
Route                      0
Distance_KM                0
Fuel_Consumption_L         0
Toll_Charges_INR           0
Traffic_Delay_Minutes      0
Weather_Impact           106
dtype: int64

In [73]:
routes_df.dtypes

Order_ID                  object
Route                     object
Distance_KM              float64
Fuel_Consumption_L       float64
Toll_Charges_INR         float64
Traffic_Delay_Minutes      int64
Weather_Impact            object
dtype: object

In [75]:
routes_df.shape

(150, 7)

## handling missing values

In [78]:
routes_df['Weather_Impact'].unique()

array([nan, 'Light_Rain', 'Fog', 'Heavy_Rain'], dtype=object)

## encoding columns

In [89]:
routes_df.isnull().sum()

Order_ID                   0
Route                      0
Distance_KM                0
Fuel_Consumption_L         0
Toll_Charges_INR           0
Traffic_Delay_Minutes      0
Weather_Impact           106
dtype: int64

In [90]:
routes_df.head()

,Order_ID,Route,Distance_KM,Fuel_Consumption_L,Toll_Charges_INR,Traffic_Delay_Minutes,Weather_Impact
0,ORD000001,Kolkata-Hyderabad,152.59,23.02,122.08,21,NaN
1,ORD000002,Hyderabad-Kolkata,362.05,43.98,289.64,33,NaN
2,ORD000003,Mumbai-Pune,519.74,65.75,415.79,2,NaN
3,ORD000004,Hyderabad-Ahmedabad,540.87,61.85,432.70,112,NaN
4,ORD000005,Chennai-Mumbai,1251.56,147.54,1001.25,10,NaN


In [92]:
def fix_routes_weather(routes_df):
    df = routes_df.copy()
    
    # FIX: Assign the result back
    df['Weather_Impact'] = df['Weather_Impact'].fillna('Normal')
    
    # Convert to categorical
    df['Weather_Impact'] = df['Weather_Impact'].astype('category')
    df['Route'] = df['Route'].astype('category')
    
    # Create features
    df['Fuel_Efficiency'] = df['Distance_KM'] / df['Fuel_Consumption_L']
    df['Cost_Per_KM'] = df['Toll_Charges_INR'] / df['Distance_KM']
    
    print(f"✅ Final NaN count: {df['Weather_Impact'].isna().sum()}")
    print(f"✅ Unique values: {df['Weather_Impact'].unique()}")
    
    return df

# Apply the fix
routes_df = fix_routes_weather(routes_df)

✅ Final NaN count: 0
✅ Unique values: ['Normal', 'Light_Rain', 'Fog', 'Heavy_Rain']
Categories (4, object): ['Fog', 'Heavy_Rain', 'Light_Rain', 'Normal']


In [93]:
routes_df.isnull().sum()

Order_ID                 0
Route                    0
Distance_KM              0
Fuel_Consumption_L       0
Toll_Charges_INR         0
Traffic_Delay_Minutes    0
Weather_Impact           0
Fuel_Efficiency          0
Cost_Per_KM              0
dtype: int64

In [94]:
routes_df.head()

,Order_ID,Route,Distance_KM,Fuel_Consumption_L,Toll_Charges_INR,Traffic_Delay_Minutes,Weather_Impact,Fuel_Efficiency,Cost_Per_KM
0,ORD000001,Kolkata-Hyderabad,152.59,23.02,122.08,21,Normal,6.628584,0.800052
1,ORD000002,Hyderabad-Kolkata,362.05,43.98,289.64,33,Normal,8.232151,0.800000
2,ORD000003,Mumbai-Pune,519.74,65.75,415.79,2,Normal,7.904791,0.799996
3,ORD000004,Hyderabad-Ahmedabad,540.87,61.85,432.70,112,Normal,8.744867,0.800007
4,ORD000005,Chennai-Mumbai,1251.56,147.54,1001.25,10,Normal,8.482852,0.800002


## Route pre-processing

In [95]:
def complete_routes_preprocessing(routes_df):
    df = routes_df.copy()
    
    print("=== COMPLETING ROUTES PREPROCESSING ===")
    
    # 1. Data type conversions
    df['Route'] = df['Route'].astype('category')
    df['Weather_Impact'] = df['Weather_Impact'].astype('category')
    
    # 2. Create ML features
    df['Fuel_Efficiency'] = df['Distance_KM'] / df['Fuel_Consumption_L']  # KM per liter
    df['Cost_Per_KM'] = df['Toll_Charges_INR'] / df['Distance_KM']
    
    # 3. Traffic categories
    df['Traffic_Level'] = pd.cut(df['Traffic_Delay_Minutes'],
                               bins=[-1, 10, 30, 60, float('inf')],
                               labels=['Low', 'Medium', 'High', 'Very_High'])
    
    # 4. Route complexity based on distance
    df['Route_Complexity'] = pd.cut(df['Distance_KM'],
                                  bins=[0, 200, 500, 1000, float('inf')],
                                  labels=['Short', 'Medium', 'Long', 'Very_Long'])
    
    # 5. Weather severity score
    weather_severity = {'Normal': 0, 'Light_Rain': 1, 'Fog': 2, 'Heavy_Rain': 3}
    df['Weather_Severity'] = df['Weather_Impact'].map(weather_severity)
    
    print("✅ Routes preprocessing completed!")
    print(f"Final shape: {df.shape}")
    print(f"New features created: Fuel_Efficiency, Cost_Per_KM, Traffic_Level, Route_Complexity, Weather_Severity")
    
    return df

# Apply complete preprocessing
routes_df_final = complete_routes_preprocessing(routes_df)

# Save it
routes_df_final.to_csv('cleaned_routes.csv', index=False)
print("✅ Cleaned routes saved!")

=== COMPLETING ROUTES PREPROCESSING ===
✅ Routes preprocessing completed!
Final shape: (150, 12)
New features created: Fuel_Efficiency, Cost_Per_KM, Traffic_Level, Route_Complexity, Weather_Severity
✅ Cleaned routes saved!


In [96]:
print("=== ROUTES DATASET FINAL CHECK ===")
print(f"Shape: {routes_df_final.shape}")
print(f"Missing values: {routes_df_final.isnull().sum().sum()}")
print(f"Data types:\n{routes_df_final.dtypes}")
print(f"\nWeather_Impact distribution:\n{routes_df_final['Weather_Impact'].value_counts()}")

=== ROUTES DATASET FINAL CHECK ===
Shape: (150, 12)
Missing values: 0
Data types:
Order_ID                   object
Route                    category
Distance_KM               float64
Fuel_Consumption_L        float64
Toll_Charges_INR          float64
Traffic_Delay_Minutes       int64
Weather_Impact           category
Fuel_Efficiency           float64
Cost_Per_KM               float64
Traffic_Level            category
Route_Complexity         category
Weather_Severity         category
dtype: object

Weather_Impact distribution:
Weather_Impact
Normal        106
Light_Rain     24
Heavy_Rain     14
Fog             6
Name: count, dtype: int64


In [97]:
routes_df.head()

,Order_ID,Route,Distance_KM,Fuel_Consumption_L,Toll_Charges_INR,Traffic_Delay_Minutes,Weather_Impact,Fuel_Efficiency,Cost_Per_KM
0,ORD000001,Kolkata-Hyderabad,152.59,23.02,122.08,21,Normal,6.628584,0.800052
1,ORD000002,Hyderabad-Kolkata,362.05,43.98,289.64,33,Normal,8.232151,0.800000
2,ORD000003,Mumbai-Pune,519.74,65.75,415.79,2,Normal,7.904791,0.799996
3,ORD000004,Hyderabad-Ahmedabad,540.87,61.85,432.70,112,Normal,8.744867,0.800007
4,ORD000005,Chennai-Mumbai,1251.56,147.54,1001.25,10,Normal,8.482852,0.800002


## routes data cleaned SUCCESSFULLY ✅ 

# 3

## vehicle_fleet

In [100]:
vehicle_df = pd.read_csv('vehicle_fleet.csv')
vehicle_df.head()

,Vehicle_ID,Vehicle_Type,Capacity_KG,Fuel_Efficiency_KM_per_L,Current_Location,Status,Age_Years,CO2_Emissions_Kg_per_KM
0,VEH0001,Refrigerated,1531.01,5.81,Hyderabad,Available,7.641663,0.465
1,VEH0002,Small_Van,632.66,10.41,Chennai,In_Transit,1.474560,0.259
2,VEH0003,Large_Truck,8903.66,6.13,Bangalore,Available,7.496936,0.441
3,VEH0004,Refrigerated,2545.81,7.14,Bangalore,In_Transit,1.415229,0.378
4,VEH0005,Small_Van,925.24,9.38,Delhi,In_Transit,1.964305,0.288


In [101]:
vehicle_df['Vehicle_Type'].unique()

array(['Refrigerated', 'Small_Van', 'Large_Truck', 'Medium_Truck',
       'Express_Bike'], dtype=object)

In [102]:
vehicle_df['Capacity_KG'].unique()

array([1531.01,  632.66, 8903.66, 2545.81,  925.24, 2671.22, 7176.62,
       2601.67, 2423.87, 2319.65, 2236.98,  932.65, 2252.27, 2859.13,
        695.34, 7022.64, 3355.08, 8357.4 , 9951.81,  759.35, 5949.08,
        602.67, 5719.71,  546.03, 8518.96, 6564.54,  724.54,  793.79,
       3872.53, 1782.81, 6635.22,  717.2 , 3341.35, 2608.08,  955.58,
       5405.75, 2566.61, 7607.25, 2000.41, 2181.52, 3070.05,  721.96,
       2056.62, 6301.18,   39.87,  654.9 , 5383.18,   23.07, 3807.12,
       2515.41])

In [103]:
vehicle_df['Fuel_Efficiency_KM_per_L'].unique()

array([ 5.81, 10.41,  6.13,  7.14,  9.38,  8.94,  5.65,  6.15,  7.53,
        7.9 ,  8.88,  8.63,  5.21,  5.04,  8.77,  4.21,  6.22,  5.9 ,
        4.98,  8.72,  5.75,  9.93,  5.79, 10.4 ,  5.12,  4.81, 10.45,
       10.8 ,  7.63,  7.02,  6.67, 10.89,  8.1 ,  9.42,  6.85,  8.36,
        7.6 ,  6.89,  6.43,  8.09,  6.63,  4.48, 30.81,  9.  ,  6.62,
       25.32,  6.23,  7.51])

In [104]:
vehicle_df['Current_Location'].unique()

array(['Hyderabad', 'Chennai', 'Bangalore', 'Delhi', 'Mumbai', 'Pune',
       'Ahmedabad', 'Kolkata'], dtype=object)

In [105]:
vehicle_df['Status'].unique()

array(['Available', 'In_Transit', 'Maintenance'], dtype=object)

In [106]:
vehicle_df['Status'].unique()

array(['Available', 'In_Transit', 'Maintenance'], dtype=object)

In [107]:
vehicle_df['CO2_Emissions_Kg_per_KM'].unique()

array([0.465, 0.259, 0.441, 0.378, 0.288, 0.302, 0.478, 0.439, 0.359,
       0.342, 0.304, 0.313, 0.518, 0.536, 0.308, 0.642, 0.434, 0.458,
       0.543, 0.31 , 0.47 , 0.272, 0.467, 0.26 , 0.527, 0.561, 0.258,
       0.25 , 0.354, 0.385, 0.405, 0.248, 0.333, 0.287, 0.394, 0.323,
       0.355, 0.392, 0.42 , 0.334, 0.407, 0.603, 0.088, 0.3  , 0.408,
       0.107, 0.433])

In [109]:
vehicle_df.isnull().sum()

Vehicle_ID                  0
Vehicle_Type                0
Capacity_KG                 0
Fuel_Efficiency_KM_per_L    0
Current_Location            0
Status                      0
Age_Years                   0
CO2_Emissions_Kg_per_KM     0
dtype: int64

In [110]:
vehicle_df.dtypes

Vehicle_ID                   object
Vehicle_Type                 object
Capacity_KG                 float64
Fuel_Efficiency_KM_per_L    float64
Current_Location             object
Status                       object
Age_Years                   float64
CO2_Emissions_Kg_per_KM     float64
dtype: object

In [ ]:
def preprocess_vehicles(vehicle_df):
    df = vehicle_df.copy()
    
    print("=== VEHICLE DATA PREPROCESSING ===")
    print(f"Missing values: {df.isnull().sum().sum()}")
    
    # 1. Convert categorical columns
    categorical_cols = ['Vehicle_Type', 'Current_Location', 'Status']
    for col in categorical_cols:
        df[col] = df[col].astype('category')
        print(f"{col}: {df[col].nunique()} categories")
    
    # 2. Create useful features
    df['Efficiency_Score'] = df['Fuel_Efficiency_KM_per_L'] / df['CO2_Emissions_Kg_per_KM']
    df['Age_Category'] = pd.cut(df['Age_Years'],
                              bins=[0, 2, 5, 10, float('inf')],
                              labels=['New', 'Young', 'Middle', 'Old'])
    
    # 3. Vehicle capacity categories
    df['Capacity_Category'] = pd.cut(df['Capacity_KG'],
                                   bins=[0, 1000, 5000, float('inf')],
                                   labels=['Small', 'Medium', 'Large'])
    
    print("✅ Vehicle preprocessing completed!")
    print(f"New features: Efficiency_Score, Age_Category, Capacity_Category")
    
    return df

# Apply preprocessing
vehicle_df_cleaned = preprocess_vehicles(vehicle_df)

# Save it
vehicle_df_cleaned.to_csv('cleaned_vehicles.csv', index=False)
print("✅ cleaned_vehicles.csv saved!")

=== VEHICLE DATA PREPROCESSING ===
Missing values: 0
Vehicle_Type: 5 categories
Current_Location: 8 categories
Status: 3 categories
✅ Vehicle preprocessing completed!
New features: Efficiency_Score, Age_Category, Capacity_Category
✅ cleaned_vehicles.csv saved!


## Vehicles Cleaned SUCCESSFULLY ✅

In [116]:
cost_breakdown_df = pd.read_csv("cost_breakdown.csv")
cost_breakdown_df.head()

,Order_ID,Fuel_Cost,Labor_Cost,Vehicle_Maintenance,Insurance,Packaging_Cost,Technology_Platform_Fee,Other_Overhead
0,ORD000001,151.36,110.31,53.00,32.17,37.46,43.77,30.38
1,ORD000002,157.32,146.20,51.61,28.54,21.74,48.64,38.74
2,ORD000003,322.61,319.74,128.03,58.13,102.91,92.06,55.45
3,ORD000004,215.33,194.01,60.87,49.68,45.89,50.40,35.12
4,ORD000005,191.92,150.42,79.03,48.92,53.35,48.62,52.98


In [117]:
cost_breakdown_df.dtypes

Order_ID                    object
Fuel_Cost                  float64
Labor_Cost                 float64
Vehicle_Maintenance        float64
Insurance                  float64
Packaging_Cost             float64
Technology_Platform_Fee    float64
Other_Overhead             float64
dtype: object

In [118]:
cost_breakdown_df.isnull().sum()

Order_ID                   0
Fuel_Cost                  0
Labor_Cost                 0
Vehicle_Maintenance        0
Insurance                  0
Packaging_Cost             0
Technology_Platform_Fee    0
Other_Overhead             0
dtype: int64

In [119]:
def preprocess_cost_breakdown(cost_df):
    df = cost_df.copy()
    
    print("=== COST BREAKDOWN PREPROCESSING ===")
    print(f"Missing values: {df.isnull().sum().sum()}")
    print(f"Shape: {df.shape}")
    
    # 1. Create total cost column
    cost_columns = ['Fuel_Cost', 'Labor_Cost', 'Vehicle_Maintenance', 'Insurance', 
                   'Packaging_Cost', 'Technology_Platform_Fee', 'Other_Overhead']
    
    df['Total_Cost'] = df[cost_columns].sum(axis=1)
    
    # 2. Create cost percentages
    for col in cost_columns:
        df[f'{col}_Percentage'] = (df[col] / df['Total_Cost']) * 100
    
    # 3. Create cost categories
    df['Cost_Category'] = pd.cut(df['Total_Cost'],
                               bins=[0, 500, 1000, 2000, float('inf')],
                               labels=['Low', 'Medium', 'High', 'Very_High'])
    
    # 4. Identify primary cost driver
    df['Primary_Cost_Driver'] = df[cost_columns].idxmax(axis=1)
    
    print("✅ Cost breakdown preprocessing completed!")
    print(f"Total cost stats: Min=${df['Total_Cost'].min():.2f}, Max=${df['Total_Cost'].max():.2f}, Avg=${df['Total_Cost'].mean():.2f}")
    print(f"New features: Total_Cost, Cost_Percentages, Cost_Category, Primary_Cost_Driver")
    
    return df

# Apply preprocessing
cost_df_cleaned = preprocess_cost_breakdown(cost_breakdown_df)

# Save it
cost_df_cleaned.to_csv('cleaned_cost_breakdown.csv', index=False)
print("✅ cleaned_cost_breakdown.csv saved!")

=== COST BREAKDOWN PREPROCESSING ===
Missing values: 0
Shape: (150, 8)
✅ Cost breakdown preprocessing completed!
Total cost stats: Min=$170.99, Max=$1322.50, Avg=$630.12
New features: Total_Cost, Cost_Percentages, Cost_Category, Primary_Cost_Driver
✅ cleaned_cost_breakdown.csv saved!


## cost breakdown cleaned SUCCESSFULLY ✅

In [120]:
cust_feedback_df = pd.read_csv('customer_feedback.csv')
cust_feedback_df.head()

,Order_ID,Feedback_Date,Rating,Feedback_Text,Would_Recommend,Issue_Category
0,ORD000002,2025-10-03,1,"Great service, very fast delivery!",Yes,Timing
1,ORD000003,2025-09-30,3,"Perfect condition, thank you",No,Timing
2,ORD000007,2025-10-12,4,"Delayed by 3 days, not acceptable",Yes,Timing
3,ORD000009,2025-09-25,5,Wrong item delivered,Yes,Service
4,ORD000011,2025-09-16,3,"Perfect condition, thank you",No,Timing


In [121]:
cust_feedback_df.dtypes

Order_ID           object
Feedback_Date      object
Rating              int64
Feedback_Text      object
Would_Recommend    object
Issue_Category     object
dtype: object

In [122]:
cust_feedback_df.isnull().sum()

Order_ID            0
Feedback_Date       0
Rating              0
Feedback_Text       0
Would_Recommend     0
Issue_Category     28
dtype: int64

In [123]:
cust_feedback_df.shape

(83, 6)

In [125]:
def preprocess_customer_feedback(cust_feedback_df):
    df = cust_feedback_df.copy()
    
    print("=== CUSTOMER FEEDBACK PREPROCESSING ===")
    print(f"Missing values: {df.isnull().sum()}")
    
    # 1. Handle missing Issue_Category
    df['Issue_Category'] = df['Issue_Category'].fillna('No_Issue')
    
    # 2. Convert Feedback_Date to datetime
    df['Feedback_Date'] = pd.to_datetime(df['Feedback_Date'])
    
    # 3. Create numeric scores FIRST before converting to categorical
    df['Recommend_Score'] = df['Would_Recommend'].map({'Yes': 1, 'No': 0})
    
    # 4. Create composite satisfaction score
    df['Satisfaction_Score'] = (df['Rating'] * 0.7) + (df['Recommend_Score'] * 0.3 * 5)
    
    # 5. NOW convert categorical columns
    categorical_cols = ['Would_Recommend', 'Issue_Category']
    for col in categorical_cols:
        df[col] = df[col].astype('category')
        print(f"{col}: {df[col].nunique()} categories")
    
    # 6. Create satisfaction level
    df['Satisfaction_Level'] = pd.cut(df['Rating'],
                                    bins=[0, 2, 3, 5],
                                    labels=['Low', 'Medium', 'High'])
    
    # 7. Extract time-based features
    df['Feedback_DayOfWeek'] = df['Feedback_Date'].dt.day_name()
    df['Feedback_Month'] = df['Feedback_Date'].dt.month
    
    print("✅ Customer feedback preprocessing completed!")
    print(f"Rating distribution:\n{df['Rating'].value_counts().sort_index()}")
    print(f"Issue categories:\n{df['Issue_Category'].value_counts()}")
    print(f"Satisfaction Score range: {df['Satisfaction_Score'].min():.1f} to {df['Satisfaction_Score'].max():.1f}")
    
    return df

# Apply preprocessing
cust_feedback_cleaned = preprocess_customer_feedback(cust_feedback_df)

# Save it
cust_feedback_cleaned.to_csv('cleaned_customer_feedback.csv', index=False)
print("✅ cleaned_customer_feedback.csv saved!")

=== CUSTOMER FEEDBACK PREPROCESSING ===
Missing values: Order_ID            0
Feedback_Date       0
Rating              0
Feedback_Text       0
Would_Recommend     0
Issue_Category     28
dtype: int64
Would_Recommend: 2 categories
Issue_Category: 5 categories
✅ Customer feedback preprocessing completed!
Rating distribution:
Rating
1    12
2     3
3    23
4    13
5    32
Name: count, dtype: int64
Issue categories:
Issue_Category
No_Issue    28
Timing      23
Quality     17
Service     12
Other        3
Name: count, dtype: int64
Satisfaction Score range: 0.7 to 5.0
✅ cleaned_customer_feedback.csv saved!


## customer feedback cleaned SUCCESSFULLY ✅ 

In [126]:
warehouse_inventory_df = pd.read_csv('warehouse_inventory.csv')
warehouse_inventory_df.head()

,Warehouse_ID,Location,Product_Category,Current_Stock_Units,Reorder_Level,Storage_Cost_per_Unit,Last_Restocked_Date
0,WH001_Mumbai,Mumbai,Electronics,3042,925,18.32,2025-10-06
1,WH001_Mumbai,Mumbai,Fashion,1800,798,13.07,2025-10-07
2,WH001_Mumbai,Mumbai,Food & Beverage,4690,823,10.29,2025-10-04
3,WH001_Mumbai,Mumbai,Healthcare,3418,815,43.83,2025-10-16
4,WH001_Mumbai,Mumbai,Industrial,745,819,28.17,2025-09-29


In [127]:
warehouse_inventory_df.dtypes

Warehouse_ID              object
Location                  object
Product_Category          object
Current_Stock_Units        int64
Reorder_Level              int64
Storage_Cost_per_Unit    float64
Last_Restocked_Date       object
dtype: object

In [128]:
warehouse_inventory_df.shape

(35, 7)

In [129]:
warehouse_inventory_df.isnull().sum()

Warehouse_ID             0
Location                 0
Product_Category         0
Current_Stock_Units      0
Reorder_Level            0
Storage_Cost_per_Unit    0
Last_Restocked_Date      0
dtype: int64

In [132]:
def preprocess_warehouse_inventory(warehouse_inventory_df):
    df = warehouse_inventory_df.copy()
    
    print("=== WAREHOUSE INVENTORY PREPROCESSING ===")
    print(f"Shape: {df.shape}")
    print(f"Missing values:\n{df.isnull().sum()}")
    
    # 1. Convert Last_Restocked_Date to datetime
    df['Last_Restocked_Date'] = pd.to_datetime(df['Last_Restocked_Date'])
    
    # 2. Convert categorical columns
    categorical_cols = ['Warehouse_ID', 'Location', 'Product_Category']
    for col in categorical_cols:
        df[col] = df[col].astype('category')
        print(f"{col}: {df[col].nunique()} categories")
    
    # 3. Create inventory management features
    df['Stock_Status'] = pd.cut(df['Current_Stock_Units'] - df['Reorder_Level'],
                              bins=[-float('inf'), 0, float('inf')],
                              labels=['Below_Reorder', 'Above_Reorder'])
    
    df['Utilization_Rate'] = df['Current_Stock_Units'] / df['Reorder_Level']
    
    # 4. Days since last restock
    current_date = pd.Timestamp.now()
    df['Days_Since_Restock'] = (current_date - df['Last_Restocked_Date']).dt.days
    
    # 5. Inventory value
    df['Inventory_Value'] = df['Current_Stock_Units'] * df['Storage_Cost_per_Unit']
    
    # 6. Create urgency score
    df['Restock_Urgency'] = np.where(
        df['Stock_Status'] == 'Below_Reorder',
        (df['Reorder_Level'] - df['Current_Stock_Units']) / df['Reorder_Level'],
        0
    )
    
    print("✅ Warehouse inventory preprocessing completed!")
    print(f"Stock Status:\n{df['Stock_Status'].value_counts()}")
    print(f"Average Utilization Rate: {df['Utilization_Rate'].mean():.2f}")
    print(f"Average Days Since Restock: {df['Days_Since_Restock'].mean():.1f}")
    
    return df

# Apply preprocessing
warehouse_df_cleaned = preprocess_warehouse_inventory(warehouse_inventory_df)

# Save it
warehouse_df_cleaned.to_csv('cleaned_warehouse_inventory.csv', index=False)
print("✅ cleaned_warehouse_inventory.csv saved!")

=== WAREHOUSE INVENTORY PREPROCESSING ===
Shape: (35, 7)
Missing values:
Warehouse_ID             0
Location                 0
Product_Category         0
Current_Stock_Units      0
Reorder_Level            0
Storage_Cost_per_Unit    0
Last_Restocked_Date      0
dtype: int64
Warehouse_ID: 5 categories
Location: 5 categories
Product_Category: 7 categories
✅ Warehouse inventory preprocessing completed!
Stock Status:
Stock_Status
Above_Reorder    28
Below_Reorder     7
Name: count, dtype: int64
Average Utilization Rate: 4.12
Average Days Since Restock: 26.7
✅ cleaned_warehouse_inventory.csv saved!


## warehouse inventory cleaned SUCCESSFULLY ✅ 